In [ ]:
dbutils.widgets.text("catalog", "adb_adventureworks_project")
dbutils.widgets.text("gold_schema", "gold_dev")

#### How do revenue, order volume, quantity sold, and estimated gross profit change over time?

In [0]:
%sql

CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :gold_schema || '.agg_monthly_sales')
USING DELTA
AS
SELECT
    d.Year,
    d.Month,
    d.YearMonth,

    COUNT(DISTINCT f.OrderNumber) AS TotalOrders,
    
    SUM(f.OrderQuantity) AS UnitsSold,
    SUM(f.EstimatedRevenue) AS EstimatedRevenue,
    SUM(f.EstimatedTotalCost) AS EstimatedTotalCost,
    SUM(f.EstimatedGrossProfit) AS EstimatedGrossProfit,

    ROUND(SUM(f.EstimatedRevenue) / NULLIF(COUNT(DISTINCT f.OrderNumber), 0), 2) AS AverageOrderValue,

    ROUND(100.0 * SUM(f.EstimatedGrossProfit) / NULLIF(SUM(f.EstimatedRevenue), 0), 2) AS GrossMarginPct

FROM 
    IDENTIFIER(:catalog || '.' || :gold_schema || '.fact_sales') f

JOIN 
    IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_date') d
ON 
    TO_DATE(f.OrderDate) = d.FullDate
GROUP BY
    d.Year,
    d.Month,
    d.YearMonth;


## Visualization

### How has Adventure Works' estimated sales revenue changed over time?

In [0]:
%sql

SELECT
    YearMonth,
    ROUND(EstimatedRevenue, 2) AS MonthlyRevenue
FROM 
    adb_adventureworks_project.gold.agg_monthly_sales
ORDER BY 
    YearMonth ASC;

Databricks visualization. Run in Databricks to view.

#### Is revenue increasing because Adventure Works receives more orders, or because customers spend more per order?

In [0]:
%sql

SELECT
    YearMonth,
    TotalOrders,
    ROUND(EstimatedRevenue, 2) AS EstimatedRevenue,
    AverageOrderValue
FROM 
    adb_adventureworks_project.gold.agg_monthly_sales
ORDER BY 
    YearMonth;

Databricks visualization. Run in Databricks to view.

#### Did customers start buying more lower-priced products after August 2016?

In [0]:
%sql

SELECT
    CASE
        WHEN f.OrderDate < '2016-08-01'
            THEN 'Before Aug 2016'
        ELSE 'Aug 2016 onward'
    END AS SalesPeriod,

    p.CategoryName,

    COUNT(DISTINCT f.OrderNumber) AS TotalOrders,
    SUM(f.OrderQuantity) AS UnitsSold,

    ROUND(SUM(f.EstimatedRevenue), 2) AS EstimatedRevenue,

    ROUND(SUM(f.EstimatedRevenue) / NULLIF(COUNT(DISTINCT f.OrderNumber), 0), 2) AS CategoryAOV

FROM 
    adb_adventureworks_project.gold.fact_sales f
JOIN 
    adb_adventureworks_project.gold.dim_product p
ON 
    f.ProductKey = p.ProductKey
GROUP BY
    CASE
        WHEN f.OrderDate < '2016-08-01'
            THEN 'Before Aug 2016'
        ELSE 'Aug 2016 onward'
    END,
    p.CategoryName
ORDER BY
    p.CategoryName,
    SalesPeriod;


#### Which product categories contributed to the jump in orders between July and August 2016?

In [0]:
%sql

SELECT
    DATE_FORMAT(f.OrderDate, 'yyyy-MM') AS YearMonth,
    p.CategoryName,

    COUNT(DISTINCT f.OrderNumber) AS TotalOrders,
    SUM(f.OrderQuantity) AS UnitsSold,

    ROUND(SUM(f.EstimatedRevenue), 2) AS EstimatedRevenue
FROM 
    adb_adventureworks_project.gold.fact_sales f
JOIN 
    adb_adventureworks_project.gold.dim_product p
ON 
    f.ProductKey = p.ProductKey
WHERE 
    f.OrderDate >= '2016-07-01' AND f.OrderDate < '2016-09-01'
GROUP BY
    DATE_FORMAT(f.OrderDate, 'yyyy-MM'),
    p.CategoryName
ORDER BY
    YearMonth,
    CategoryName;
